# 02 — Transformer Block → Tiny GPT

**Goal (about 3 sessions):** build every remaining piece of a decoder-only transformer, assemble a tiny GPT, and prove it all works by overfitting one sentence until the model reproduces it perfectly.

**Same rules as notebook 01:** no AI completion, a test cell after each `TODO` (✅ ❌ ⏳ 💥), and if you're stuck for more than 15 minutes, write down what's confusing and ask for a hint.

**Plan and suggested pacing**

| Session | Parts | Minimum | Target |
|---|---|---|---|
| 1 | 0–1 | your MHA pasted in + LayerNorm green | both norms green |
| 2 | 2–3 | MLP green | transformer block green |
| 3 | 4–5 | TinyGPT tests green | overfit test green + wrap-up out loud |

A CPU runtime is enough for this whole notebook.

In [8]:
import sys, math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device}")
torch.manual_seed(0)

torch 2.14.1+cpu | device: cpu


In [2]:
# Test runner — run once.
def run_tests(*tests):
    for t in tests:
        name = t.__name__.removeprefix("test_")
        try:
            t()
            print(f"✅ {name}")
        except NotImplementedError:
            print(f"⏳ {name}: not implemented yet")
        except NameError as e:
            print(f"⏳ {name}: {e} — implemented / pasted yet?")
        except AssertionError as e:
            print(f"❌ {name}: {e}")
        except Exception as e:
            print(f"💥 {name}: {type(e).__name__}: {e}")

## Part 0 — Bring in your attention from notebook 01

Paste **your** `scaled_dot_product_attention` and `MultiHeadAttention` (the versions that passed every test) into the next cell. You'll build on your own code from here on.

In [6]:
# ⬇️ Paste your scaled_dot_product_attention and MultiHeadAttention from notebook 01 here.

def softmax_torch(x, axis=-1):
    # TODO: numerically stable softmax along `axis`
    e = torch.exp(x - torch.amax(x, axis=axis, keepdim=True))
    res = e / torch.sum(e, axis=axis, keepdim=True)
    return res



def scaled_dot_product_attention(q, k, v, mask=None):
    # TODO: same four steps as the NumPy version, in torch. Return (output, weights).
    qk = q @ k.swapaxes(-1, -2) / torch.sqrt(torch.tensor(k.shape[-1]))
    #  2. if mask is given, block positions where mask is False
    if mask is not None:
      qk = torch.where(mask, qk, -np.inf)
    #  3. weights = softmax over the *keys* axis
    weights = softmax_torch(qk)
    #  4. output = weights @ V
    output = weights @ v
    return output, weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model, self.n_heads, self.d_head = d_model, n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        return

    def forward(self, x, mask=None):
        # x: (B, T, d_model); mask: bool, broadcastable to (B, H, T, T), True = attend
        B, T, _ = x.shape
        #  1. project x to q, k, v
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        #  2. split heads: (B, T, d_model) -> (B, H, T, d_head)
        q = q.view(B, T, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        v = v.view(B, T, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        k = k.view(B, T, self.n_heads, self.d_head).permute(0, 2, 1, 3)

        #  3. attention with your scaled_dot_product_attention
        output, weights = scaled_dot_product_attention(q, k, v, mask=mask)

        #  4. merge heads: (B, H, T, d_head) -> (B, T, d_model)
        output = output.transpose(2,1).contiguous().view(B, T,self.d_model)

        #  5. output projection W_o
        output = self.W_o(output)
        return output

In [7]:
def test_pasted_mha_matches_pytorch():
    torch.manual_seed(0)
    d_model, n_heads, B, T = 32, 4, 2, 7
    x = torch.randn(B, T, d_model)
    causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
    m = MultiHeadAttention(d_model, n_heads).eval()
    ref = nn.MultiheadAttention(d_model, n_heads, batch_first=True).eval()
    with torch.no_grad():
        ref.in_proj_weight.copy_(torch.cat([m.W_q.weight, m.W_k.weight, m.W_v.weight]))
        ref.in_proj_bias.copy_(torch.cat([m.W_q.bias, m.W_k.bias, m.W_v.bias]))
        ref.out_proj.weight.copy_(m.W_o.weight); ref.out_proj.bias.copy_(m.W_o.bias)
    ref_out, _ = ref(x, x, x, attn_mask=~causal, need_weights=False)
    out = m(x, mask=causal)
    assert torch.allclose(out, ref_out, atol=1e-5), f"max diff {(out - ref_out).abs().max():.2e}"

run_tests(test_pasted_mha_matches_pytorch)

⏳ pasted_mha_matches_pytorch: name 'np' is not defined — implemented / pasted yet?


## Part 1 — Normalization

### 1a. LayerNorm

For each token's vector **separately** (over the last dimension):

$$\text{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta$$

where μ and σ² are that vector's mean and variance, and γ (`self.weight`) and β (`self.bias`) are learnable vectors of size `d`, initialised to ones and zeros.

- Learnable tensors must be wrapped in `nn.Parameter(...)`, or PyTorch won't train them.
- **Trap:** LayerNorm uses the *biased* variance (divide by n, not n−1). Check what `torch.var` does by default.

In [44]:
class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        # TODO: self.weight (ones, shape (d,)) and self.bias (zeros, shape (d,)) as nn.Parameter

        self.weight = nn.Parameter(torch.ones(d))
        self.bias = nn.Parameter(torch.zeros(d))

    def forward(self, x):
        # TODO: normalize over the last dim, then scale by weight and shift by bias
        mu = torch.mean(x, dim=-1, keepdim=True)
        std_x = x.var( dim=-1, keepdim=True, unbiased=False)


        x_no = (x - mu) / torch.sqrt(std_x + self.eps)

        return self.weight * x_no + self.bias


In [45]:
def test_layernorm_matches_pytorch():
    torch.manual_seed(0)
    x = torch.randn(2, 5, 16) * 3 + 1
    ln, ref = LayerNorm(16), nn.LayerNorm(16)
    with torch.no_grad():
        ln.weight.copy_(torch.randn(16)); ln.bias.copy_(torch.randn(16))
        ref.weight.copy_(ln.weight); ref.bias.copy_(ln.bias)
    diff = (ln(x) - ref(x)).abs().max()
    assert diff < 1e-5, f"max diff {diff:.2e} — biased vs unbiased variance?"

def test_layernorm_params_registered():
    n = sum(p.numel() for p in LayerNorm(16).parameters())
    assert n == 32, f"{n} learnable parameters, expected 32 (weight + bias) — wrapped in nn.Parameter?"

run_tests(test_layernorm_matches_pytorch, test_layernorm_params_registered)

✅ layernorm_matches_pytorch
✅ layernorm_params_registered


### 1b. RMSNorm (the modern default, used in LLaMA-family models)

RMSNorm drops the mean-centring and the bias. It just rescales each vector by its root-mean-square:

$$\text{RMSNorm}(x) = \gamma \odot \frac{x}{\sqrt{\text{mean}(x^2) + \varepsilon}}$$

Only `self.weight` (ones, shape `(d,)`), and no bias.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        # TODO: self.weight (ones, shape (d,)) as nn.Parameter
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
def test_rmsnorm_unit_rms():
    x = torch.randn(3, 4, 32) * 5
    out = RMSNorm(32)(x)
    rms = out.pow(2).mean(-1).sqrt()
    assert torch.allclose(rms, torch.ones_like(rms), atol=1e-3), f"RMS of outputs should be ≈1, got {rms.flatten()[:3]}"

def test_rmsnorm_scale_invariant():
    x = torch.randn(2, 3, 16)
    n = RMSNorm(16)
    assert torch.allclose(n(x), n(7.0 * x), atol=1e-5), "RMSNorm(c·x) should equal RMSNorm(x)"

def test_rmsnorm_no_mean_centring():
    x = torch.full((1, 1, 8), 2.0)  # constant vector: LayerNorm would give 0, RMSNorm gives 1
    assert torch.allclose(RMSNorm(8)(x), torch.ones(1, 1, 8), atol=1e-4), "a constant vector should map to ones"

def test_rmsnorm_matches_pytorch():
    if not hasattr(nn, "RMSNorm"):
        print("   (nn.RMSNorm not in this torch version — skipped)"); return
    torch.manual_seed(0)
    x = torch.randn(2, 5, 16)
    n, ref = RMSNorm(16), nn.RMSNorm(16, eps=1e-6)
    with torch.no_grad():
        n.weight.copy_(torch.randn(16)); ref.weight.copy_(n.weight)
    diff = (n(x) - ref(x)).abs().max()
    assert diff < 1e-5, f"max diff {diff:.2e}"

run_tests(test_rmsnorm_unit_rms, test_rmsnorm_scale_invariant,
          test_rmsnorm_no_mean_centring, test_rmsnorm_matches_pytorch)

**Questions (write your answers in LOG.md, then send them to Claude for feedback):**
1. Transformers use LayerNorm, while CNNs traditionally use BatchNorm. Name two reasons BatchNorm is a poor fit for sequence models.
2. RMSNorm drops the mean-centring and the bias. What does that save, and why does it barely hurt quality?

## Part 2 — The MLP (feed-forward) sublayer

Each token goes through the same small two-layer network **independently**: expand to `4 × d_model`, apply a nonlinearity, project back.

**Naming (the tests depend on it):** `self.fc1 = nn.Linear(d_model, hidden_mult * d_model)`, `self.fc2 = nn.Linear(hidden_mult * d_model, d_model)`, with GELU between them (`F.gelu`).

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_model, hidden_mult=4):
        super().__init__()
        # TODO: self.fc1, self.fc2
        raise NotImplementedError

    def forward(self, x):
        # TODO: fc1 -> GELU -> fc2
        raise NotImplementedError

In [ ]:
def test_mlp_shape_and_params():
    m = MLP(32)
    x = torch.randn(2, 5, 32)
    assert m(x).shape == (2, 5, 32), f"shape {tuple(m(x).shape)}"
    n = sum(p.numel() for p in m.parameters())
    expected = (32 * 128 + 128) + (128 * 32 + 32)
    assert n == expected, f"{n} parameters, expected {expected}"

def test_mlp_is_per_token():
    # The MLP must not mix information between tokens
    m = MLP(16); x = torch.randn(1, 6, 16)
    full = m(x)
    one_at_a_time = torch.cat([m(x[:, t:t+1]) for t in range(6)], dim=1)
    assert torch.allclose(full, one_at_a_time, atol=1e-6), "output for a token depends on other tokens"

def test_mlp_uses_gelu():
    m = MLP(16); x = torch.randn(2, 3, 16)
    assert torch.allclose(m(x), m.fc2(F.gelu(m.fc1(x))), atol=1e-6), "expected fc2(gelu(fc1(x)))"

run_tests(test_mlp_shape_and_params, test_mlp_is_per_token, test_mlp_uses_gelu)

**Question:** attention mixes information *across* tokens, and the MLP processes each token *alone*. In one sentence each, what is each sublayer for? Then: in a block with `d_model = d`, which sublayer has more parameters, and by roughly how much?

## Part 3 — The transformer block (pre-norm)

$$x \leftarrow x + \text{Attn}(\text{Norm}_1(x)) \qquad x \leftarrow x + \text{MLP}(\text{Norm}_2(x))$$

This is the **pre-norm** arrangement used by GPT-2 onwards: normalize *before* each sublayer, add the result back onto the unnormalized `x` (the residual stream).

**Naming:** `self.norm1`, `self.attn` (your `MultiHeadAttention`), `self.norm2`, `self.mlp`. Use your `RMSNorm` or `LayerNorm`, whichever you prefer.

One of the tests zeroes the output layers of both sublayers and checks that the block becomes the identity. Before running it, think about why that holds for pre-norm but would **fail** for post-norm, `x ← Norm(x + Sublayer(x))`.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        # TODO: self.norm1, self.attn, self.norm2, self.mlp
        raise NotImplementedError

    def forward(self, x, mask=None):
        # TODO: two pre-norm residual sublayers
        raise NotImplementedError

In [ ]:
d_model, n_heads, B, T = 32, 4, 2, 7
x = torch.randn(B, T, d_model)
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))

def test_block_shape():
    assert TransformerBlock(d_model, n_heads)(x, causal).shape == (B, T, d_model)

def test_block_identity_when_outputs_zeroed():
    blk = TransformerBlock(d_model, n_heads)
    with torch.no_grad():
        for layer in (blk.attn.W_o, blk.mlp.fc2):
            layer.weight.zero_(); layer.bias.zero_()
    out = blk(x, causal)
    assert torch.allclose(out, x, atol=1e-6), "block should be the identity — residual wiring or post-norm?"

def test_block_is_causal():
    blk = TransformerBlock(d_model, n_heads).eval()
    x2 = x.clone(); x2[:, -1] = torch.randn(B, d_model)
    a, b = blk(x, causal), blk(x2, causal)
    assert torch.allclose(a[:, :-1], b[:, :-1], atol=1e-6), "information leaked from the future"

def test_block_all_params_get_gradients():
    blk = TransformerBlock(d_model, n_heads)
    blk(x, causal).pow(2).mean().backward()
    missing = [n for n, p in blk.named_parameters() if p.grad is None or p.grad.abs().sum() == 0]
    assert not missing, f"no gradient for: {missing}"

run_tests(test_block_shape, test_block_identity_when_outputs_zeroed,
          test_block_is_causal, test_block_all_params_get_gradients)

**Question (good one for your optimization background):** in a deep pre-norm stack, follow the gradient from the loss back to the input of the *first* block. What does the residual stream guarantee about that path, and why does it make deep transformers easier to train than post-norm ones? The original 2017 Transformer used post-norm, so what did it need to compensate?

## Part 4 — TinyGPT

Put it together. Input: integer token ids `idx` of shape `(B, T)`. Output: next-token logits `(B, T, vocab_size)`.

**Naming:**
- `self.tok_emb = nn.Embedding(vocab_size, d_model)`: a vector per token id
- `self.pos_emb = nn.Embedding(max_len, d_model)`: a learned vector per position (modern models use RoPE instead; that comes later)
- `self.blocks`: `n_layers` × `TransformerBlock`, stored in an **`nn.ModuleList`**, not a Python list. Know why.
- `self.norm_f`: a final norm
- `self.head = nn.Linear(d_model, vocab_size)`

**forward:** token embeddings + position embeddings for positions `0 … T−1` → build the `(T, T)` causal mask → run the blocks → final norm → head.

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, max_len=128):
        super().__init__()
        self.max_len = max_len
        # TODO: tok_emb, pos_emb, blocks (nn.ModuleList), norm_f, head
        raise NotImplementedError

    def forward(self, idx):
        # idx: (B, T) integer tensor -> logits (B, T, vocab_size)
        # TODO
        raise NotImplementedError

In [ ]:
V = 20

def test_gpt_shape():
    m = TinyGPT(V)
    idx = torch.randint(0, V, (3, 10))
    assert m(idx).shape == (3, 10, V), f"logits shape {tuple(m(idx).shape)}, expected (3, 10, {V})"

def test_gpt_uses_modulelist():
    m = TinyGPT(V, n_layers=3)
    assert isinstance(m.blocks, nn.ModuleList), "store blocks in nn.ModuleList — otherwise their parameters are invisible to the optimizer"
    assert len(m.blocks) == 3

def test_gpt_is_causal():
    m = TinyGPT(V).eval()
    idx = torch.randint(0, V, (1, 12))
    idx2 = idx.clone(); idx2[0, -1] = (idx[0, -1] + 1) % V
    a, b = m(idx), m(idx2)
    assert torch.allclose(a[:, :-1], b[:, :-1], atol=1e-5), "logits at earlier positions changed when only the last token changed"

run_tests(test_gpt_shape, test_gpt_uses_modulelist, test_gpt_is_causal)

## Part 5 — Prove it works: overfit one sentence

If the whole model is correct, it should be able to **memorize** a single short sentence: loss → ≈ 0, and generation reproduces it exactly. This is the first sanity check you should run on any new model (Karpathy's "A Recipe for Training Neural Networks"). If a model *can't* overfit one example, something is broken, and there's no point training on real data yet.

The data prep below is done for you: a character-level vocabulary, with `x` = the sentence and `y` = the same sentence shifted by one character.

In [ ]:
text = "the quick brown fox jumps over the lazy dog."
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))
x_train, y_train = data[:-1].unsqueeze(0), data[1:].unsqueeze(0)   # (1, T)
vocab_size = len(chars)
print(f"vocab size: {vocab_size} | sequence length: {x_train.shape[1]}")
print("x:", repr(decode(x_train[0].tolist()))[:30], "...")
print("y:", repr(decode(y_train[0].tolist()))[:30], "...")

### The training loop

Write the body of the loop: forward → loss → zero the gradients → backward → optimizer step.

- `F.cross_entropy` expects logits of shape `(N, C)` and targets of shape `(N,)`. Your logits are `(B, T, V)` and your targets `(B, T)`, so reshape accordingly.
- Append `loss.item()` to `losses` each step.

In [ ]:
def train_overfit(model, x, y, steps=300, lr=3e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for step in range(steps):
        # TODO: forward, cross-entropy loss, zero_grad, backward, step; append loss.item()
        raise NotImplementedError
    return losses

In [ ]:
@torch.no_grad()
def generate_greedy(model, prompt, n_new):
    idx = torch.tensor([encode(prompt)])
    for _ in range(n_new):
        logits = model(idx[:, -model.max_len:])
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return decode(idx[0].tolist())

model, losses = None, None

def test_initial_loss_is_about_log_vocab():
    global model, losses
    torch.manual_seed(0)
    model = TinyGPT(vocab_size)
    losses = train_overfit(model, x_train, y_train)
    expected = math.log(vocab_size)
    assert abs(losses[0] - expected) < 1.0, f"initial loss {losses[0]:.2f}, expected ≈ ln({vocab_size}) = {expected:.2f}"

def test_overfits_to_near_zero():
    if losses is None: raise NotImplementedError
    assert losses[-1] < 0.05, f"final loss {losses[-1]:.3f} — should memorize one sentence (< 0.05)"

def test_generates_the_sentence():
    if model is None or losses is None: raise NotImplementedError
    model.eval()
    out = generate_greedy(model, text[0], len(text) - 1)
    assert out == text, f"generated {out!r}"

run_tests(test_initial_loss_is_about_log_vocab, test_overfits_to_near_zero, test_generates_the_sentence)
if losses:
    print(f"\nloss: step 0 = {losses[0]:.3f} → step {len(losses)-1} = {losses[-1]:.4f}")

In [ ]:
# Plot the loss curve
import matplotlib.pyplot as plt
if losses:
    plt.plot(losses); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss (log scale)")
    plt.title("Overfitting one sentence"); plt.show()
else:
    print("⏳ Run the training tests first.")

**Questions:**
1. Why should the loss at step 0 be close to ln(vocab_size)? What would a much larger starting loss tell you?
2. If the loss plateaued at, say, 1.5 and never went lower, list three bugs you'd check first.
3. Memorizing one sentence proves very little about *learning*. So what exactly does it prove?

## Wrap-up — self-check (out loud, no notes)

1. Draw a pre-norm block and trace the tensor shapes for a `(B, T)` input through TinyGPT to the logits.
2. Parameter count of one block in terms of `d` (attention + MLP + norms). Which part dominates?
3. Pre-norm vs post-norm: what changes, and why does pre-norm train more stably?
4. Why `nn.ModuleList` rather than a Python list?
5. What does a modern LLM do differently from this TinyGPT? (Hint: think positions, the MLP's activation, norms on q and k, and what's cached during generation.)

**Before you close Colab:** three lines in `LOG.md`, then **File → Save a copy in GitHub**.

**Next (notebook 03):** train on a real text dataset, sample with temperature, and log curves to wandb.